# Making the Linear Kalman Filter Bulletproof

*Course 2 — Linear Kalman Filter, Part 3. The textbook KF of [08](08_Deriving_the_Linear_Kalman_Filter.ipynb) assumes perfect arithmetic and perfectly white, zero-mean, uncorrelated noise. This notebook removes those crutches: **numerical robustness**, the **square-root KF**, **initialization/tuning**, **bias estimation**, and **correlated / colored noise**.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 Numerical Robustness — Keep the Covariance Symmetric & Positive-Definite

- Within the KF, both covariances must stay **symmetric** and **positive-definite** (all eigenvalues $>0$) at every step. Floating-point round-off can violate both.

  → If $\Sigma$ silently loses symmetry or gains a negative eigenvalue, the confidence bounds become meaningless and the filter can **diverge**.

- The time update $\Sigma_{\tilde{x},k}^{-} = A\,\Sigma_{\tilde{x},k-1}^{+}A^{T} + \Sigma_{\tilde{w}}$ adds two PD quantities, so it is safe. The danger is the measurement update $\Sigma_{\tilde{x},k}^{+} = \Sigma_{\tilde{x},k}^{-} - L_k C_k \Sigma_{\tilde{x},k}^{-}$ — a **subtraction** that can go non-PD.

---

### 🧩 The Joseph-Form Covariance Update

- Replace step 2c with the algebraically-equivalent **Joseph form**:

$$
\Sigma_{\tilde{x},k}^{+} = (I - L_k C_k)\,\Sigma_{\tilde{x},k}^{-}\,(I - L_k C_k)^{T} + L_k\,\Sigma_{\tilde{v}}\,L_k^{T}.
$$

  → Both terms are "squared" (of the form $M\Sigma M^T$), so each is guaranteed positive-semidefinite — we **add** two PD quantities instead of subtracting. Symmetry and positivity are preserved by construction, at modest extra cost.

- **→ Intuition:** same answer in exact arithmetic, far more forgiving in floating point. A cheap insurance policy every practical KF should carry.

### 🧩 Higham's Nearest-SPD Fix (last resort)

- If a computed $\Sigma$ *still* comes out slightly non-positive-definite, project it to the **nearest symmetric positive-semidefinite matrix**:

$$
\Sigma = U S V^T \ (\text{SVD}), \quad H = V S V^T, \quad \Sigma \leftarrow \tfrac{1}{4}\big(\Sigma + \Sigma^T + H + H^T\big).
$$

  → Symmetrize and "fold in" the SVD-based PSD companion $H$; the average is the closest valid covariance in Frobenius norm.

```octave
[~,S,V] = svd(SigmaX);
H = V*S*V';
SigmaX = (SigmaX + SigmaX' + H + H')/4;
```

- **→ Intuition:** Joseph form *prevents* most problems; Higham's method *repairs* the rare survivor. Together they keep the filter's uncertainty legal.

### 🧩 The Square-Root Kalman Filter — Why Square Roots?

- Consider representing a covariance vs. its Cholesky factor. The numbers $\{10^6, 10^2, 1\}$ span **six** orders of magnitude; their square roots $\{10^3, 10^1, 1\}$ span only **three**.

  → Propagating the **Cholesky factor** $\mathcal{S}$ (where $\Sigma = \mathcal{S}\mathcal{S}^T$) instead of $\Sigma$ **halves the dynamic range** (condition number), so single-precision arithmetic can do what would otherwise need double.

- Bonus: a factored covariance is **positive-definite by construction** — you literally cannot represent a negative eigenvalue.

- **→ Intuition:** the SR-KF is the KF rewritten to carry $\mathcal{S}$ everywhere. Steps that don't touch covariance (1a, 1c, 2b) are unchanged; only the covariance steps (1b, 2a, 2c) are recast in factor form.

### 🧩 SR-KF: The Covariance Steps in Factor Form

- Define lower-triangular Cholesky factors $\Sigma_{\tilde{x}}^{\pm} = \mathcal{S}_{\tilde{x}}^{\pm}(\mathcal{S}_{\tilde{x}}^{\pm})^{T}$ and $\Sigma_{\tilde{w}} = \mathcal{S}_{\tilde{w}}\mathcal{S}_{\tilde{w}}^T$, $\Sigma_{\tilde{v}} = \mathcal{S}_{\tilde{v}}\mathcal{S}_{\tilde{v}}^T$.

- **Step 1b (prediction covariance)** via **QR decomposition**: stack the two contributions and QR them.

$$
\mathcal{S}_{\tilde{x},k}^{-} = \operatorname{qr}\!\Big(\big[\; A_{k-1}\mathcal{S}_{\tilde{x},k-1}^{+}, \;\; \mathcal{S}_{\tilde{w}}\;\big]^{T}\Big)^{T}
$$

  → We want the factor of $A\Sigma^+A^T + \Sigma_{\tilde w} = MM^T$. Since $M=[A\mathcal S^+,\ \mathcal S_{\tilde w}]$ is $n\times 2n$, take its **QR**: the upper-triangular $R$ is the Cholesky factor we need. No explicit $\Sigma$ ever formed.

- **Step 2a (gain)**: get $\mathcal{S}_{\tilde{z},k}$ by QR of $[\,C_k\mathcal{S}_{\tilde{x},k}^{-},\ \mathcal{S}_{\tilde{v}}\,]$, then solve for $L_k$ by back-substitution (no matrix inverse).

- **Step 2c (estimation covariance)** via **Cholesky downdating** (`cholupdate(...,'-')`): downdate $\mathcal{S}_{\tilde{x}}^{-}$ by each column of $L_k\mathcal{S}_{\tilde{z}}$.

  → The measurement update *subtracts*, so QR (which builds up) won't work — instead **downdate** the factor column by column, the numerically-careful way to shrink a Cholesky factor.

```octave
Sminus = qr([Ad*Splus, Sw]')';  Sminus = tril(Sminus(1:nx,1:nx));   % 1b
Sz = qr([Cd*Sminus, Sv]')';  Sz = tril(Sz(1:nz,1:nz));
L = (Sminus*Sminus')*Cd'/Sz'/Sz;                                     % 2a
cov_update = L*Sz;
for j = 1:length(zhat)
  Sminus = cholupdate(Sminus, cov_update(:,j), '-');                 % 2c
end
Splus = Sminus';
```

- **→ Intuition:** results are **indistinguishable** from the standard KF in double precision, but the SR-KF stays healthy where the plain filter would break — essential for embedded/fixed-point implementations.

### 🧩 Initialization

- A KF must be started with:

$$
\hat{x}_0^{+} = \mathbb{E}[x_0], \qquad \Sigma_{\tilde{x},0}^{+} = \mathbb{E}\!\big[(x_0-\hat{x}_0^{+})(x_0-\hat{x}_0^{+})^{T}\big].
$$

  → The initial best guess of the state, and how uncertain that guess is.

- We rarely know these exactly. Two options: (1) **guess** — the KF converges toward truth even from a poor start, so use whatever domain knowledge you have; (2) fuse a few pre-"time-zero" measurements (better in theory, usually not worth the complexity).

- **→ Intuition:** guessing $\hat x_0^+$ is intuitive; guessing $\Sigma_{\tilde x,0}^+$ is less so — set its diagonal so the true state plausibly lies within $\hat x_0^+ \pm 3\sqrt{\operatorname{diag}(\Sigma_{\tilde x,0}^+)}$. Start it **too large** rather than too small: an over-confident start (tiny $\Sigma_0$) makes the filter ignore early measurements and converge slowly.

### 🧩 Tuning $\Sigma_{\tilde{w}}$ and $\Sigma_{\tilde{v}}$ (and convergence rate)

- $\Sigma_{\tilde{w}}$ (process-noise covariance) drives the state in $x_{k+1}=Ax_k+Bu_k+w_k$; $\Sigma_{\tilde{v}}$ (sensor-noise covariance) corrupts $z_k = Cx_k+Du_k+v_k$.

- **No model is perfect**, so both are also used to absorb *model error*, not just physical noise. Set them **larger** than the raw noise to account for inaccuracies in $A,B,C,D$.

- Their **ratio** controls behavior:

  → Large $\Sigma_{\tilde{w}}$ → "trust the sensor over the model" → wide bounds, **fast** convergence.
  → Large $\Sigma_{\tilde{v}}$ → "trust the model over the sensor" → narrow bounds, **slow** convergence (pseudo open-loop).

- **→ Intuition:** tuning is largely picking this trust ratio. Model inaccuracy is hard to quantify, so trial-and-error is normal — and you **cannot** get arbitrarily fast convergence *and* arbitrarily tight bounds at once. (Course 3 adds *adaptive* filters that tune $\Sigma_{\tilde w},\Sigma_{\tilde v}$ online.)

### 🧩 Nonzero-Mean Noise → Bias Estimation (Friedland)

- The KF assumes $\mathbb{E}[w_k]=\mathbb{E}[v_k]=0$. If noises have **nonzero mean** (bias), estimates and bounds degrade. Model a bias vector $b_k$:

$$
x_{k+1} = A x_k + B u_k + B^{b} b_k + \tilde{w}_k, \qquad z_k = C x_k + D u_k + C^{b} b_k + \tilde{v}_k,
$$

  with $\tilde w_k,\tilde v_k$ zero-mean/white and the bias $b_k$ (assumed slowly varying) carrying all the mean.

  → Split each noise into a **bias part** (its mean) and a **clean zero-mean part**. Now we estimate the bias explicitly and subtract its effect.

- **Friedland's decoupling:** run the **ordinary (biased) KF** *and* a **second bias-estimating filter** in parallel, then combine:

$$
\hat{x}_k^{\text{unbiased}} = \hat{x}_k^{+} + \delta_k, \qquad \Sigma_{\tilde{x},k}^{\text{unbiased}} = \Sigma_{\tilde{x},k}^{+} + \Sigma_{\tilde{\delta},k}.
$$

  → The main KF ignores the bias and is wrong by a computable amount; the bias filter estimates $\hat b_k$ and produces the correction $\delta_k$ that de-biases the state and its covariance.

- **→ Intuition:** *"don't estimate what you already know"* — if you happen to know a bias exactly, just subtract it. Friedland handles the case where the bias itself must be learned online. The main KF's equations stay **unchanged**; a bolt-on second filter does the correcting.

### 🧩 Cross-Correlated $w_k$ and $v_k$ (Coincident)

- Standard KF assumes $\mathbb{E}[w_k v_j^T]=0$. But a shared disturbance (temperature, EMI) can make process and sensor noise correlated **at the same instant**:

$$
\mathbb{E}[w_k v_k^T] = \Sigma_{\tilde{w}\tilde{v}} \neq 0.
$$

- Trick: add a cleverly-chosen zero, $x_{k+1}=Ax_k+Bu_k+w_k + T\underbrace{(z_k - C x_k - D u_k - v_k)}_{=\,0}$, and pick $T$ to kill the correlation:

$$
T = \Sigma_{\tilde{w}\tilde{v}}\,\Sigma_{\tilde{v}}^{-1}, \qquad \bar{A}_k = A_k - T C_k, \quad \Sigma_{\bar{w}} = \Sigma_{\tilde{w}} - \Sigma_{\tilde{w}\tilde{v}}\Sigma_{\tilde{v}}^{-1}\Sigma_{\tilde{w}\tilde{v}}^{T}.
$$

  → Redefine the dynamics with $\bar A_k$ and a **new process noise** $\bar w_k = w_k - Tv_k$ that is uncorrelated with $v_k$. Then the standard KF applies to the modified model, with $z_k$ folded in as an extra known input.

- A related case is **shifted** correlation ($\mathbb{E}[w_k v_j^T]=\Sigma_{\tilde w\tilde v}\delta_{k,j-1}$, one step apart), which changes only the **gain**:

$$
L_k = \big[\Sigma_{\tilde{x},k}^{-}C_k^{T} + \Sigma_{\tilde{w}\tilde{v}}\big]\big(C_k\Sigma_{\tilde{x},k}^{-}C_k^{T} + \Sigma_{\tilde{v}} + C_k\Sigma_{\tilde{w}\tilde{v}} + \Sigma_{\tilde{w}\tilde{v}}^{T}C_k^{T}\big)^{-1}.
$$

  → Same filter, gain adjusted so the correlation between the state-prediction error and the measurement noise is accounted for.

### 🧩 Auto-Correlated (Colored) Noise → State Augmentation

- The KF also assumes noises are **white**. If the *process* noise is colored, model it with a shaping filter (see [06](06_Stochastic_Processes_and_Propagating_Uncertainty.ipynb)) $w_k = A_w w_{k-1} + \bar{w}_{k-1}$ ($\bar w$ white), then **augment the state** with the noise state:

$$
\begin{bmatrix} x_k \\ w_k\end{bmatrix} = \begin{bmatrix} A_{k-1} & I \\ 0 & A_w\end{bmatrix}\begin{bmatrix} x_{k-1} \\ w_{k-1}\end{bmatrix} + \begin{bmatrix} B_{k-1} \\ 0\end{bmatrix}u_{k-1} + \begin{bmatrix} 0 \\ \bar{w}_{k-1}\end{bmatrix}.
$$

  → Fold the colored noise's own dynamics *into* the state. The augmented system is driven by **white** $\bar w_{k-1}$, so the standard KF assumptions hold again — at the price of a bigger state.

- **Colored sensor noise** $v_k = A_v v_{k-1} + \bar v_{k-1}$ is handled the same way, and the resulting zero-measurement-noise case is cleaned up with **measurement differencing** (using a synthetic $\bar z_k = z_{k+1}-A_v z_k$).

- **→ Intuition:** almost every "the KF assumption is violated" problem is solved by the same move — **augment the state** so that the *new* driving noise is white and zero-mean. You trade computation (bigger $A$) for correctness.

### 🧩 Summary

- **Numerical robustness:** use the **Joseph form** (adds two PD terms instead of subtracting) and, if needed, **Higham's** nearest-SPD projection to keep $\Sigma$ symmetric and positive-definite.

- **Square-root KF:** propagate the Cholesky factor $\mathcal S$ (halved dynamic range, PD by construction) using **QR** for time/gain updates and **Cholesky downdating** for the measurement update.

- **Initialization/tuning:** start $\Sigma_0$ generously; the ratio $\Sigma_{\tilde w}/\Sigma_{\tilde v}$ trades convergence speed against bound tightness, and both absorb model error.

- **Bias (Friedland):** run a second bias filter in parallel and correct the main (biased) KF's output.

- **Correlated/colored noise:** decorrelate with a transform $T$, adjust the gain for shifted correlation, or — the universal tool — **augment the state** so the driving noise is white again.

---
*Next: [10 · KF Extensions — Fault Detection, Sequential Processing, Steady-State, Prediction & Smoothing](10_KF_Extensions_FaultDetection_SteadyState_Smoothing.ipynb).*